## **Langchain을 표현하는 언어, LCEL**

**[LCEL로 기본 체인 구성하기]**

In [1]:
#필수 라이브러리 설치
!pip install --upgrade --quiet langchain openai langchain-core langchain-openai


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

#프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")

#LLM 호출
model = ChatOpenAI(model="gpt-4o-mini")

#출력 파서 설정
output_parser = StrOutputParser()

#LCEL로 프롬프트템플릿-LLM-출력 파서 연결하기
chain = prompt | model | output_parser

#invoke함수로 chain 실행하기
chain.invoke({"topic": "ice cream"})

/Users/jinhohyeon/Desktop/dev/Learned/llm/udemy-genairag-langchain-chatbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'What did the ice cream cone say to the ice cream sundae? \n\n"I think you\'re sweet, but I can\'t handle your toppings!"'

**[Streaming 기능 추가를 더욱 쉽게, stream()]**

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

#Chain 선언
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = prompt | model

#Chain의 stream()함수를 통해 스트리밍 기능 추가
for s in chain.stream({"topic": "bears"}):
    print(s.content, end="", flush=True)

Why do bears have hairy coats?

Because they look silly in sweaters!

**[한꺼번에 여러 개 API 요청하고 답변 받기, batch()]**

- 5개 문장 번역 batch 수행

In [4]:
%%time
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("다음 한글 문장을 프랑스어로 번역해줘 {sentence}")
chain = prompt | model

chain.batch([
    {"sentence": "그녀는 매일 아침 책을 읽습니다."},
    {"sentence": "오늘 날씨가 참 좋네요."},
    {"sentence": "저녁에 친구들과 영화를 볼 거예요."},
    {"sentence": "그 학생은 매우 성실하게 공부합니다."},
    {"sentence": "커피 한 잔이 지금 딱 필요해요."}

])

CPU times: user 26.5 ms, sys: 6.45 ms, total: 33 ms
Wall time: 1.56 s


[AIMessage(content='그녀는 매일 아침 책을 읽습니다.라는 문장을 프랑스어로 번역하면 다음과 같습니다.\n\n**Elle lit un livre tous les matins.**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 33, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_791df6ace0', 'id': 'chatcmpl-DgSnDZln39IZTfHvYnaLm6C1LnHLn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e356d-76e2-7da3-8a5e-91a7a40fe44e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 38, 'total_tokens': 71, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 AIMessage(content="오늘 날씨가 참 좋네요.  \n=> Au

- 1개 문장 번역 invoke 수행

In [5]:
%%time
prompt = ChatPromptTemplate.from_template("다음 한글 문장을 프랑스어로 번역해줘 {sentence}")
chain = prompt | model

chain.invoke({"sentence": "그녀는 매일 아침 책을 읽습니다."})

CPU times: user 5.34 ms, sys: 1.74 ms, total: 7.08 ms
Wall time: 1.33 s


AIMessage(content='그녀는 매일 아침 책을 읽습니다.는 프랑스어로 "Elle lit un livre chaque matin."입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 33, 'total_tokens': 61, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_791df6ace0', 'id': 'chatcmpl-DgSnJNSlHmawLPF2arRRKDDNYsQNq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e356d-8ef2-7c92-9cce-11a6cb9edb17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## **RunnablePassthrough, RunnableLambda, RunnableParallel**

**[RunnablePassthrough]**

**RunnablePassthrough는 가장 단순한 Runnable 객체로, 들어온 입력을 그대로 전달합니다.**

In [6]:
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke("안녕하세요")

'안녕하세요'

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("다음 한글 문장을 프랑스어로 번역해줘 {sentence} \n French sentence: (print from here)")
model = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()

runnable_chain = {"sentence": RunnablePassthrough()} | prompt | model | output_parser

runnable_chain.invoke({"sentence": "그녀는 매일 아침 책을 읽습니다."})

'Elle lit un livre chaque matin.'

RunnablePassthrough는 assin 함수를 통해 새로운 변수에 계산된 값을 입력할 수 있습니다.

In [8]:
(RunnablePassthrough.assign(mult=lambda x: x["num"]*3)).invoke({"num":3})

{'num': 3, 'mult': 9}

In [9]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'extra': {'num': 1, 'mult': 3}, 'modified': 2}

**[RunnableLambda]**

**RunnableLambda는 임의의 함수를 Chain에 결합할 수 있게 Runnable 객체로 변환합니다..**

In [10]:
def add_smile(x):
    return x + ":)"

In [11]:
from langchain_core.runnables import RunnableLambda

add_smile = RunnableLambda(add_smile)

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt_str = "{topic}의 역사에 대해 세문장으로 설명해주세요."
prompt = ChatPromptTemplate.from_template(prompt_str)

model = ChatOpenAI(model_name = 'gpt-4o-mini')

output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [16]:
from langchain_core.runnables import RunnableLambda

def add_thank(x):
    return x + " 들어주셔서 감사합니다 :)"

add_thank = RunnableLambda(add_thank)

In [17]:
chain = prompt | model | output_parser | add_thank
chain.invoke("반도체")

'반도체의 역사는 20세기 초에 시작되어, 1947년 벨 연구소에서 최초의 트랜지스터가 발명됨으로써 혁신이 시작되었습니다. 이후 1960년대에는 집적회로(IC)가 개발되어 전자기기의 소형화와 성능 향상을 이끌었습니다. 21세기 들어서는 반도체가 인공지능, 모바일, IoT 등 다양한 분야에서 핵심 기술로 자리 잡으며 산업 전반에 큰 영향을 미치고 있습니다. 들어주셔서 감사합니다 :)'

In [18]:
!pip install -q grandalf


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [19]:
chain.get_graph().print_ascii()

    +-------------+    
    | PromptInput |    
    +-------------+    
           *           
           *           
           *           
+--------------------+ 
| ChatPromptTemplate | 
+--------------------+ 
           *           
           *           
           *           
    +------------+     
    | ChatOpenAI |     
    +------------+     
           *           
           *           
           *           
  +-----------------+  
  | StrOutputParser |  
  +-----------------+  
           *           
           *           
           *           
    +-----------+      
    | add_thank |      
    +-----------+      
           *           
           *           
           *           
 +------------------+  
 | add_thank_output |  
 +------------------+  


**[RunnableParallel]**

**RunnableParallel은 여러 요소가 병렬 처리되도록 처리합니다.**

In [20]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'modified': 2}

In [21]:
runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    modified=add_thank,
)

In [22]:
runnable.invoke("안녕하세요")

{'passed': '안녕하세요', 'modified': '안녕하세요 들어주셔서 감사합니다 :)'}

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model = 'gpt-4o-mini', max_tokens = 128, temperature = 0)

history_prompt = ChatPromptTemplate.from_template("{topic}가 무엇의 약자인지 알려주세요.")
celeb_prompt = ChatPromptTemplate.from_template("{topic} 분야의 유명인사 3명의 이름만 알려주세요.")

output_parser = StrOutputParser()

history_chain = history_prompt | model | output_parser
celeb_chain = celeb_prompt | model | output_parser

map_chain = RunnableParallel(history=history_chain, celeb=celeb_chain)

result = map_chain.invoke({"topic": "AI"})

In [25]:
result

{'history': 'AI는 "Artificial Intelligence"의 약자로, 한국어로는 "인공지능"이라고 합니다. 인공지능은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 문제 해결 등의 작업을 수행할 수 있도록 하는 기술을 의미합니다.',
 'celeb': '1. 앤드류 응 (Andrew Ng)\n2. 제프리 힌튼 (Geoffrey Hinton)\n3. 얀 르쿤 (Yann LeCun)'}

## **LCEL로 RAG 구축하기**

**[기본 RAG 구축]**

In [34]:
!pip install langchainhub


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from langchain_openai import ChatOpenAI
import bs4
from langchainhub.client import Client

hub = Client()

from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

model = ChatOpenAI(model="gpt-4o-mini")
# Load, chunk and index the contents of the blog.
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/mnews/article/366/0001007619",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("newsct_article _article_body")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# Retrieve and generate using the relevant snippets of the blog.
retriever = vectorstore.as_retriever()
from langchain_core.load import loads

prompt = loads(hub.pull("rlm/rag-prompt"))


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)
rag_chain.invoke("아까 내가 했던 질문이 뭐야?")

/var/folders/sd/72757lvs4fx3jpxplj70p5nh0000gn/T/ipykernel_90855/2126862631.py:34: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = loads(hub.pull("rlm/rag-prompt"))
/Users/jinhohyeon/Desktop/dev/Learned/llm/udemy-genairag-langchain-chatbot/.venv/lib/python3.11/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)
/var/folders/sd/72757lvs4fx3jpxplj70p5nh0000gn/T/ipykernel_90855/2126862631.py:34: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(hub.pull("rlm/rag-prompt"))
/var/folders/sd/72757lvs4fx3jpxplj70p5nh0000gn/T/ipykernel_90855/2126862631.py:34: LangChainPendingDeprecationWarning: The default value of `allo

'저는 그 질문을 알지 못합니다.'

## **메모리를 추가한 RAG 구축하기**

In [13]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

def create_history_aware_retriever(llm, retriever, prompt):
    def retrieve(x):
        if not x.get("chat_history"):
            query = x["input"]
        else:
            query = (prompt | llm | StrOutputParser()).invoke(x)
        return retriever.invoke(query)
    return RunnableLambda(retrieve)

def create_stuff_documents_chain(llm, prompt):
    return (
        RunnablePassthrough.assign(
            context=lambda x: "\n\n".join(doc.page_content for doc in x["context"])
        )
        | prompt
        | llm
        | StrOutputParser()
    )

def create_retrieval_chain(history_aware_retriever, combine_docs_chain):
    def run(x):
        docs = history_aware_retriever.invoke(x)
        answer = combine_docs_chain.invoke({**x, "context": docs})
        return {**x, "context": docs, "answer": answer}
    return RunnableLambda(run)

loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/mnews/article/366/0001007619",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("newsct_article _article_body")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

In [14]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [15]:
history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)

In [16]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
question_answer_chain = create_stuff_documents_chain(model, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [17]:

chat_history = []

question = "오픈AI 근황은 어때?"
ai_msg_1 = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), ai_msg_1["answer"]])
ai_msg_1

{'input': '오픈AI 근황은 어때?',
 'chat_history': [HumanMessage(content='오픈AI 근황은 어때?', additional_kwargs={}, response_metadata={}),
  '오픈AI는 생성형 인공지능 챗GPT를 개발하며 최근 구글에 도전장을 내밀었습니다. 이와 관련된 새로운 발전이나 기능 업데이트가 있을 수 있습니다. 하지만 구체적인 최신 소식은 확인할 수 없습니다.'],
 'context': [Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.'),
  Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.'),
  Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.'),
  Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.')],
 'answer': '오픈AI는 생성형 인공지능 챗GPT를 개발하며 최근 구글에 도전장을 내밀었습니다. 이와 관련된 새로운 발전이나 기능 업데이트가 있을 수 있습니다. 하지만 구체적인 최신 소식은 확인할 수 없습니다.'}

In [18]:
chat_history

[HumanMessage(content='오픈AI 근황은 어때?', additional_kwargs={}, response_metadata={}),
 '오픈AI는 생성형 인공지능 챗GPT를 개발하며 최근 구글에 도전장을 내밀었습니다. 이와 관련된 새로운 발전이나 기능 업데이트가 있을 수 있습니다. 하지만 구체적인 최신 소식은 확인할 수 없습니다.']

In [19]:
question = "다시 한번 설명해줄래?"
ai_msg_2 = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), ai_msg_2["answer"]])
ai_msg_2

{'input': '다시 한번 설명해줄래?',
 'chat_history': [HumanMessage(content='오픈AI 근황은 어때?', additional_kwargs={}, response_metadata={}),
  '오픈AI는 생성형 인공지능 챗GPT를 개발하며 최근 구글에 도전장을 내밀었습니다. 이와 관련된 새로운 발전이나 기능 업데이트가 있을 수 있습니다. 하지만 구체적인 최신 소식은 확인할 수 없습니다.',
  HumanMessage(content='다시 한번 설명해줄래?', additional_kwargs={}, response_metadata={}),
  '오픈AI는 생성형 인공지능 챗GPT를 개발하며 구글에 도전장을 내밀었습니다. 최근의 발전이나 기능 업데이트는 있을 수 있지만, 구체적인 최신 소식은 확인할 수 없습니다.'],
 'context': [Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.'),
  Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.'),
  Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_content='생성형 인공지능(AI) 챗GPT 개발사 오픈AI가 검색 왕국 구글에 도전장을 내밀었다.'),
  Document(metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001007619'}, page_cont

In [20]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

/Users/jinhohyeon/Desktop/dev/Learned/llm/udemy-genairag-langchain-chatbot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
conversational_rag_chain.invoke(
    {"input": "오픈AI의 근황이 어때?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'오픈AI는 생성형 인공지능 챗GPT를 개발하며 검색 왕국 구글에 도전장을 내밀고 있습니다. 이는 AI 기술 경쟁에 있어 중요한 이정표가 될 수 있습니다. 추가적인 정보는 필요하지 않습니다.'

In [22]:
conversational_rag_chain.invoke(
    {"input": "앞으로 어떻게 될까?"},
    config={"configurable": {"session_id": "abc123"}},
)["answer"]

'앞으로 오픈AI와 구글 간의 경쟁이 더욱 치열해질 것으로 예상됩니다. 인공지능 기술의 발전에 따라 다양한 응용 가능성이 열릴 것이며, 사용자들에게 더 나은 서비스가 제공될 것입니다. 하지만 특정 결과에 대해서는 확실하게 알 수 없습니다.'

In [23]:
from langchain_core.messages import AIMessage

for message in store["abc123"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

User: 오픈AI의 근황이 어때?

AI: 오픈AI는 생성형 인공지능 챗GPT를 개발하며 검색 왕국 구글에 도전장을 내밀고 있습니다. 이는 AI 기술 경쟁에 있어 중요한 이정표가 될 수 있습니다. 추가적인 정보는 필요하지 않습니다.

User: 앞으로 어떻게 될까?

AI: 앞으로 오픈AI와 구글 간의 경쟁이 더욱 치열해질 것으로 예상됩니다. 인공지능 기술의 발전에 따라 다양한 응용 가능성이 열릴 것이며, 사용자들에게 더 나은 서비스가 제공될 것입니다. 하지만 특정 결과에 대해서는 확실하게 알 수 없습니다.

